# Isotermas de van der Waals e ponto crítico
**Termodinâmica Avançada — 2026.2 · Aula de Criticalidade**

Nesta atividade vamos:
1. Traçar as isotermas $P\times v$ do CO₂ para $T$ abaixo, igual e acima de $T_c$;
2. Marcar o ponto crítico $(v_c, P_c)$;
3. Fazer a **construção de Maxwell** numa isoterma subcrítica e obter $v_\ell$ e $v_g$.

Equação de van der Waals:
$$P(v)=\frac{RT}{v-b}-\frac{a}{v^2},\qquad v_c=3b,\quad T_c=\frac{8a}{27Rb},\quad P_c=\frac{a}{27b^2}.$$

> Uso de IA liberado para ajudar no código.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Constante dos gases em L*atm/(K*mol)
R = 0.082057

# Parametros de van der Waals do CO2
a = 3.640   # L^2*atm/mol^2
b = 0.04267 # L/mol

# Ponto critico previsto por vdW
vc = 3*b
Tc = 8*a/(27*R*b)
Pc = a/(27*b**2)
print(f'vc = {vc:.4f} L/mol')
print(f'Tc = {Tc:.1f} K   (experimental: 304.2 K)')
print(f'Pc = {Pc:.1f} atm (experimental: 73.8 atm)')
print(f'Zc = Pc*vc/(R*Tc) = {Pc*vc/(R*Tc):.3f}  (vdW preve 3/8 = 0.375)')

## 1. Isotermas para várias temperaturas

In [ ]:
def P_vdw(v, T):
    return R*T/(v - b) - a/v**2

v = np.linspace(1.05*b, 0.6, 800)   # L/mol

fig, ax = plt.subplots(figsize=(7,5))
for T in [0.85*Tc, 0.90*Tc, 0.95*Tc, Tc, 1.05*Tc]:
    ax.plot(v, P_vdw(v, T), label=f'T = {T/Tc:.2f} Tc')
ax.plot(vc, Pc, 'ko', ms=8, label='ponto crítico')
ax.set_xlabel('v  (L/mol)'); ax.set_ylabel('P  (atm)')
ax.set_ylim(0, 120); ax.set_xlim(0, 0.6)
ax.set_title('Isotermas de van der Waals — CO$_2$')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

**Observe:** abaixo de $T_c$ surge o *laço de van der Waals*, com um trecho onde $(\partial P/\partial v)_T>0$ (instável). Em $T=T_c$ o laço degenera num ponto de inflexão horizontal. Acima de $T_c$ a isoterma é monotônica (fluido supercrítico).

## 2. Construção de Maxwell
Na coexistência, líquido e vapor têm o mesmo $\mu$ (mesmo $G$ molar). Isso equivale a escolher $P_\text{sat}$ de modo que as áreas do laço acima e abaixo da reta sejam iguais:
$$\int_{v_\ell}^{v_g}\!\big(P_\text{vdw}(v)-P_\text{sat}\big)\,dv = 0.$$

In [ ]:
from scipy.optimize import brentq
from scipy.integrate import quad

T = 0.90*Tc   # isoterma subcritica

# 1) Extremos do laco (espinodais): raizes de dP/dv = 0
dPdv = lambda v: -R*T/(v - b)**2 + 2*a/v**3
grid = np.linspace(1.001*b, 1.0, 5000)
roots = [brentq(dPdv, grid[i], grid[i+1])
         for i in range(len(grid)-1) if dPdv(grid[i])*dPdv(grid[i+1]) < 0]
va, vb = min(roots), max(roots)                     # espinodais
P_top = max(P_vdw(va, T), P_vdw(vb, T))              # topo do laco
P_bot = max(min(P_vdw(va, T), P_vdw(vb, T)), 1e-3)   # base do laco

# 2) Para um dado Psat, as raizes de P_vdw(v)=Psat: vl (menor) e vg (maior)
def roots_at(Psat):
    f = lambda v: P_vdw(v, T) - Psat
    vs = [brentq(f, grid[i], grid[i+1])
          for i in range(len(grid)-1) if f(grid[i])*f(grid[i+1]) < 0]
    return min(vs), max(vs)

# 3) Maxwell: escolher Psat (entre base e topo do laco) que anula a area
def area(Psat):
    vl, vg = roots_at(Psat)
    return quad(lambda v: P_vdw(v, T) - Psat, vl, vg)[0]

Psat = brentq(area, P_bot + 1e-3, P_top - 1e-3)
vl, vg = roots_at(Psat)
print(f'T = {T:.1f} K = {T/Tc:.2f} Tc')
print(f'P_sat = {Psat:.2f} atm')
print(f'v_liquido = {vl:.4f} L/mol   v_vapor = {vg:.4f} L/mol')

In [ ]:
vv = np.linspace(1.02*b, 0.6, 800)
fig, ax = plt.subplots(figsize=(7,5))
ax.plot(vv, P_vdw(vv, T), label='isoterma vdW')
ax.hlines(Psat, vl, vg, color='crimson', lw=2, label='patamar de Maxwell')
ax.plot([vl, vg], [Psat, Psat], 'ko')
ax.annotate('$v_\\ell$', (vl, Psat), textcoords='offset points', xytext=(-5,8))
ax.annotate('$v_g$', (vg, Psat), textcoords='offset points', xytext=(0,8))
ax.set_xlabel('v  (L/mol)'); ax.set_ylabel('P  (atm)')
ax.set_ylim(0, 90); ax.set_xlim(0, 0.6)
ax.set_title(f'Construção de Maxwell  (T = {T/Tc:.2f} Tc)')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

## Para entregar
1. Repita a construção de Maxwell para $T=0.80\,T_c$ e $T=0.95\,T_c$. O que acontece com a largura do patamar $(v_g-v_\ell)$ ao aproximar de $T_c$?
2. Monte a curva de coexistência (pontos $v_\ell$ e $v_g$ vs $T$) — o *domo* de coexistência.
3. **Discussão:** como $(v_g-v_\ell)$ se comporta perto de $T_c$? Relacione com o expoente crítico $\beta$ visto em aula.